In [1]:
from pyspark.sql import SparkSession, functions as F

In [2]:
spark = SparkSession.builder.appName("01_bronze").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")   # las horas se leen tal como vienen

BUCKET = "taxis-chicago-grandes-datos-506301"
taxi = spark.read.parquet(f"gs://{BUCKET}/bronze/taxi_trips")
weather = spark.read.option("mergeSchema", "true").parquet(f"gs://{BUCKET}/bronze/weather")

26/10/02 19:02:25 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
taxi.printSchema()
print(f"Viajes en bronze: {taxi.count():,}")
taxi.groupBy(F.year("trip_start_timestamp").alias("year")).count().orderBy("year").show()

root
 |-- unique_key: string (nullable = true)
 |-- taxi_id: string (nullable = true)
 |-- trip_start_timestamp: timestamp (nullable = true)
 |-- trip_end_timestamp: timestamp (nullable = true)
 |-- trip_seconds: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- pickup_census_tract: long (nullable = true)
 |-- dropoff_census_tract: long (nullable = true)
 |-- pickup_community_area: long (nullable = true)
 |-- dropoff_community_area: long (nullable = true)
 |-- fare: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- extras: double (nullable = true)
 |-- trip_total: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- company: string (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_location: string (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- drop

Viajes en bronze: 212,046,737


+----+--------+
|year|   count|
+----+--------+
|2013|35438190|
|2014|39523076|
|2015|23604750|
|2016|22707263|
|2017|19582632|
|2018|19206406|
|2019|24994550|
|2020| 5894649|
|2021| 7767266|
|2022| 6793419|
|2023| 6534536|
+----+--------+



In [4]:
weather.printSchema()
print(f"Filas de clima en bronze: {weather.count():,}")
weather.groupBy("year").count().orderBy("year").show()

root
 |-- stn: string (nullable = true)
 |-- wban: string (nullable = true)
 |-- year: string (nullable = true)
 |-- mo: string (nullable = true)
 |-- da: string (nullable = true)
 |-- temp: double (nullable = true)
 |-- count_temp: long (nullable = true)
 |-- dewp: double (nullable = true)
 |-- count_dewp: long (nullable = true)
 |-- slp: double (nullable = true)
 |-- count_slp: long (nullable = true)
 |-- stp: double (nullable = true)
 |-- count_stp: long (nullable = true)
 |-- visib: double (nullable = true)
 |-- count_visib: long (nullable = true)
 |-- wdsp: string (nullable = true)
 |-- count_wdsp: string (nullable = true)
 |-- mxpsd: string (nullable = true)
 |-- gust: double (nullable = true)
 |-- max: double (nullable = true)
 |-- flag_max: string (nullable = true)
 |-- min: double (nullable = true)
 |-- flag_min: string (nullable = true)
 |-- prcp: double (nullable = true)
 |-- flag_prcp: string (nullable = true)
 |-- sndp: double (nullable = true)
 |-- fog: string (nullable =

Filas de clima en bronze: 45,256,966


+----+-------+
|year|  count|
+----+-------+
|2013|4008161|
|2014|4115582|
|2015|4200939|
|2016|4292866|
|2017|4293349|
|2018|4010814|
|2019|4156054|
|2020|4119205|
|2021|4026550|
|2022|3994699|
|2023|4038747|
+----+-------+



In [6]:
spark.stop()